# Ch03 — 機率基礎與條件機率 (Probability Fundamentals & Conditional Probability)

**預估時間：** 1.5 小時

## 學習目標

1. 理解機率的三種定義（古典、頻率、主觀）並能正確計算事件機率
2. 熟練運用加法法則 (Addition Rule) 與乘法法則 (Multiplication Rule) 解決複合事件問題
3. 掌握條件機率 (Conditional Probability) 的計算與直覺解讀
4. 理解貝氏定理 (Bayes' Theorem) 的推導邏輯，並能應用於實務資料分析

## 前置要求

- **Ch01** 敘述統計與資料探索
- **Ch02** 常見機率分佈

請確認已完成上述章節的學習。

In [ ]:
import pandas as pd
import numpy as np

---
## Part A — 機率基本概念 (Basic Probability Concepts)

### 核心術語

| 術語 | 英文 | 說明 |
|------|------|------|
| 隨機實驗 | Random Experiment | 結果無法事先確定的實驗（例：擲硬幣、抽樣調查） |
| 樣本空間 | Sample Space, $S$ | 所有可能結果的集合（例：擲骰子 $S=\{1,2,3,4,5,6\}$） |
| 事件 | Event, $A$ | 樣本空間的子集合（例：「點數 > 4」$=\{5,6\}$） |

### 三種機率定義

1. **古典機率 (Classical Probability)**：當所有基本結果等可能時，$P(A) = \dfrac{|A|}{|S|}$
2. **頻率機率 (Frequentist Probability)**：大量重複實驗中，事件發生的相對頻率趨近之值
3. **主觀機率 (Subjective Probability)**：根據經驗、信念對事件發生可能性的主觀評估

> 在資料分析中，我們最常使用的是**頻率機率**——直接從資料中計算相對頻率。

In [ ]:
# 載入 Titanic 資料集
df = pd.read_csv("datasets/titanic_train.csv")
print(f"資料筆數: {len(df)}")
print(f"欄位: {list(df.columns)}")
df.head(3)

In [ ]:
# 計算 P(Survived=1)：存活的頻率機率
n_survived = df["Survived"].sum()
n_total = len(df)
p_survived = n_survived / n_total

print(f"存活人數: {n_survived}")
print(f"總人數:   {n_total}")
print(f"P(Survived=1) = {n_survived}/{n_total} = {p_survived:.4f}")
print(f"P(Survived=0) = {n_total - n_survived}/{n_total} = {1 - p_survived:.4f}")
print()
print("驗證: P(Survived=0) + P(Survived=1) =", round(p_survived + (1 - p_survived), 4))

### 口訣：機率三公理 (Kolmogorov Axioms)

1. **非負性**：$P(A) \geq 0$，任何事件的機率不為負
2. **正規性**：$P(S) = 1$，樣本空間的機率為 1
3. **可加性**：互斥事件 $A \cap B = \emptyset \Rightarrow P(A \cup B) = P(A) + P(B)$

> 記憶口訣：「非負、歸一、互斥可加」

**餘事件 (Complement)**：$P(A^c) = 1 - P(A)$，這是正規性的直接推論。

---
## Part B — 機率法則 (Probability Rules)

### 加法法則 (Addition Rule)

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

- 若 $A$ 與 $B$ **互斥 (Mutually Exclusive)**：$P(A \cap B) = 0$，則 $P(A \cup B) = P(A) + P(B)$

### 乘法法則 (Multiplication Rule)

$$P(A \cap B) = P(A) \cdot P(B|A) = P(B) \cdot P(A|B)$$

- 若 $A$ 與 $B$ **獨立 (Independent)**：$P(B|A) = P(B)$，則 $P(A \cap B) = P(A) \cdot P(B)$

### 互斥 vs 獨立——常見混淆

| | 互斥 (Mutually Exclusive) | 獨立 (Independent) |
|---|---|---|
| 定義 | $P(A \cap B) = 0$ | $P(A \cap B) = P(A) \cdot P(B)$ |
| 直覺 | 不能同時發生 | 一個事件不影響另一個 |
| 例子 | 擲一次骰子同時出現 1 和 6 | 擲兩顆骰子，各自的結果 |
| 能否同時成立？ | 除非 $P(A)=0$ 或 $P(B)=0$，否則**不能**同時互斥又獨立 |

> 關鍵區別：互斥是「結構性限制」（不可共存），獨立是「資訊性關係」（互不影響）。

In [ ]:
# 建立列聯表 (Contingency Table)
ct = pd.crosstab(df["Sex"], df["Survived"], margins=True)
ct.columns = ["Dead", "Survived", "Total"]
ct.index = ["Female", "Male", "Total"]
print("=== 性別 vs 存活 列聯表 ===")
ct

In [ ]:
# 從列聯表計算各機率
n = len(df)

# 個別機率
p_female = (df["Sex"] == "female").sum() / n
p_survived = df["Survived"].sum() / n

# 交集機率
p_female_and_survived = ((df["Sex"] == "female") & (df["Survived"] == 1)).sum() / n

# 加法法則: P(female OR survived)
p_female_or_survived = p_female + p_survived - p_female_and_survived

print(f"P(Female)                = {p_female:.4f}")
print(f"P(Survived)              = {p_survived:.4f}")
print(f"P(Female AND Survived)   = {p_female_and_survived:.4f}")
print(f"P(Female OR Survived)    = {p_female_or_survived:.4f}")
print()

# 驗證加法法則
p_direct = ((df["Sex"] == "female") | (df["Survived"] == 1)).sum() / n
print(f"直接計算 P(F or S)       = {p_direct:.4f}")
print(f"加法法則結果一致: {np.isclose(p_female_or_survived, p_direct)}")

In [ ]:
# 驗證獨立性: 若獨立，P(F and S) 應 = P(F) * P(S)
p_if_independent = p_female * p_survived

print("=== 獨立性檢驗 ===")
print(f"若獨立: P(F)*P(S)        = {p_if_independent:.4f}")
print(f"實際:   P(F and S)       = {p_female_and_survived:.4f}")
print(f"差異:                      {abs(p_if_independent - p_female_and_survived):.4f}")
print()
print(f"結論: 性別與存活{'不' if abs(p_if_independent - p_female_and_survived) > 0.01 else ''}獨立")
print("(實際交集機率遠大於獨立假設下的期望值，說明女性確實更容易存活)")

### 知識補給站

> **為什麼性別與存活不獨立？**
> 
> Titanic 沉船時遵循「婦女與兒童優先」(Women and Children First) 的救生原則，
> 因此性別強烈影響了存活機率。這正是「不獨立」的經典案例——
> 知道一個人的性別，會**改變**我們對其存活機率的估計。

---
## Part C — 條件機率 (Conditional Probability)

### 定義

在事件 $B$ 已發生的條件下，事件 $A$ 發生的機率：

$$P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

直覺解讀：將樣本空間**縮小**到 $B$ 發生的範圍內，再計算 $A$ 出現的比例。

In [ ]:
# P(Survived=1 | Female) vs P(Survived=1 | Male)
p_survived_given_female = df.loc[df["Sex"] == "female", "Survived"].mean()
p_survived_given_male = df.loc[df["Sex"] == "male", "Survived"].mean()

print("=== 條件機率計算 ===")
print(f"P(Survived=1 | Female) = {p_survived_given_female:.4f}")
print(f"P(Survived=1 | Male)   = {p_survived_given_male:.4f}")
print(f"差異倍數:                {p_survived_given_female / p_survived_given_male:.1f}x")
print()

# 手動驗算: P(Survived|Female) = P(Survived AND Female) / P(Female)
p_manual = p_female_and_survived / p_female
print(f"手動驗算: P(S and F)/P(F) = {p_female_and_survived:.4f} / {p_female:.4f} = {p_manual:.4f}")
print(f"與 .mean() 結果一致: {np.isclose(p_survived_given_female, p_manual)}")

### 條件機率的鏈式法則 (Chain Rule)

條件機率可以推廣到多個事件：

$$P(A \cap B \cap C) = P(A) \cdot P(B|A) \cdot P(C|A \cap B)$$

這在處理多個條件交叉時非常有用，例如同時考慮性別、艙等對存活的影響。

### 解讀

- 女性的存活率約 **74%**，男性僅約 **19%**
- 條件機率清楚揭示了：**知道性別這個條件資訊後，存活機率會大幅改變**
- 這就是條件機率的核心價值——量化「額外資訊」帶來的影響

> 如果 $P(A|B) = P(A)$，代表 $B$ 的發生不影響 $A$，此時 $A$ 與 $B$ 獨立。
> 反之，$P(A|B) \neq P(A)$ 代表兩者**相依 (Dependent)**。

---
## Part D — 貝氏定理 (Bayes' Theorem)

### 推導

從條件機率的定義出發：

$$P(A|B) = \frac{P(A \cap B)}{P(B)} \quad \text{且} \quad P(B|A) = \frac{P(A \cap B)}{P(A)}$$

兩式合併，得到**貝氏定理**：

$$\boxed{P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}}$$

### 語義角色

| 符號 | 名稱 | 意義 |
|------|------|------|
| $P(A)$ | 先驗機率 (Prior) | 在觀察到資料 $B$ 之前，對 $A$ 的信念 |
| $P(B|A)$ | 似然 (Likelihood) | 若 $A$ 為真，觀察到 $B$ 的機率 |
| $P(A|B)$ | 後驗機率 (Posterior) | 觀察到 $B$ 之後，對 $A$ 的更新信念 |
| $P(B)$ | 邊際機率 (Marginal / Evidence) | 正規化常數，確保後驗機率加總為 1 |

> **Prior + Data = Posterior**：貝氏定理的核心精神是「根據新資料更新信念」。

### 全機率公式 (Law of Total Probability)

當事件 $B_1, B_2, \ldots, B_k$ 構成樣本空間的**完整分割** (Partition) 時：

$$P(A) = \sum_{i=1}^{k} P(A|B_i) \cdot P(B_i)$$

此公式常用於貝氏定理的分母計算，將邊際機率 $P(A)$ 展開為各條件路徑的加權和。

In [ ]:
# 全機率公式驗證:
# P(Survived) = P(Survived|Female)*P(Female) + P(Survived|Male)*P(Male)
p_male = (df["Sex"] == "male").sum() / n

p_survived_total_prob = (
    p_survived_given_female * p_female +
    p_survived_given_male * p_male
)

print("=== 全機率公式驗證 ===")
print(f"P(Survived|F)*P(F) = {p_survived_given_female:.4f} * {p_female:.4f} = {p_survived_given_female * p_female:.4f}")
print(f"P(Survived|M)*P(M) = {p_survived_given_male:.4f} * {p_male:.4f} = {p_survived_given_male * p_male:.4f}")
print(f"加總                = {p_survived_total_prob:.4f}")
print(f"直接計算 P(Survived) = {p_survived:.4f}")
print(f"結果一致: {np.isclose(p_survived_total_prob, p_survived)}")

In [ ]:
# 貝氏定理實例: P(Female | Survived=1)
# 問題: 已知某人存活，此人為女性的機率是多少？

# 方法一：直接從資料計算
p_female_given_survived = (
    (df["Sex"] == "female") & (df["Survived"] == 1)
).sum() / df["Survived"].sum()

# 方法二：套用貝氏定理
# P(Female|Survived) = P(Survived|Female) * P(Female) / P(Survived)
p_bayes = (p_survived_given_female * p_female) / p_survived

print("=== 貝氏定理: P(Female | Survived=1) ===")
print(f"直接計算:   {p_female_given_survived:.4f}")
print(f"貝氏定理:   {p_bayes:.4f}")
print(f"兩者一致:   {np.isclose(p_female_given_survived, p_bayes)}")
print()
print("拆解:")
print(f"  Prior:      P(Female)             = {p_female:.4f}")
print(f"  Likelihood: P(Survived|Female)    = {p_survived_given_female:.4f}")
print(f"  Evidence:   P(Survived)            = {p_survived:.4f}")
print(f"  Posterior:  P(Female|Survived)     = {p_bayes:.4f}")

### 貝氏定理的直覺

- **先驗 (Prior)**：女性佔全體旅客約 35%
- **後驗 (Posterior)**：在「已存活」的條件下，女性佔比提升至約 68%
- 「存活」這個證據大幅更新了我們對「此人為女性」的信念

$$\underbrace{P(\text{Female})}_{\sim 0.35} \xrightarrow{\text{觀察到存活}} \underbrace{P(\text{Female}|\text{Survived})}_{\sim 0.68}$$

### 知識補給站 — Naive Bayes 在機器學習 (Machine Learning) 的應用

貝氏定理是 **Naive Bayes 分類器** 的理論基礎：

$$P(\text{class}|\text{features}) \propto P(\text{class}) \prod_{i} P(\text{feature}_i|\text{class})$$

**「Naive」的由來**：假設各特徵在給定類別下**條件獨立** (Conditionally Independent)。
雖然此假設在現實中很少完全成立，但 Naive Bayes 在以下場景表現優異：

- 垃圾郵件偵測 (Spam Detection)
- 文本分類 (Text Classification)
- 情感分析 (Sentiment Analysis)

其優勢在於：訓練速度快、不需要大量資料、對高維度特徵空間表現穩健。

---
## Part E — 實務案例分析 (Practical Case Study)

讓我們用列聯表深入分析 Titanic 資料中不同群體的存活機率。

In [ ]:
# 性別 vs 存活：條件機率表（按列正規化）
ct_sex = pd.crosstab(df["Sex"], df["Survived"], normalize="index")
ct_sex.columns = ["P(Dead|Sex)", "P(Survived|Sex)"]
print("=== 各性別的存活條件機率 ===")
print(ct_sex.round(4))
print()

In [ ]:
# 艙等 vs 存活：條件機率表
ct_class = pd.crosstab(df["Pclass"], df["Survived"], normalize="index")
ct_class.columns = ["P(Dead|Pclass)", "P(Survived|Pclass)"]
print("=== 各艙等的存活條件機率 ===")
print(ct_class.round(4))

In [ ]:
# 交叉分析: 艙等 x 性別 vs 存活
ct_cross = pd.crosstab(
    [df["Pclass"], df["Sex"]], df["Survived"], normalize="index"
)
ct_cross.columns = ["P(Dead)", "P(Survived)"]
print("=== 艙等 x 性別的存活條件機率 ===")
print(ct_cross.round(4))
print()

# 找出存活率最高與最低的組合
best = ct_cross["P(Survived)"].idxmax()
worst = ct_cross["P(Survived)"].idxmin()
print(f"存活率最高: Pclass={best[0]}, Sex={best[1]} -> {ct_cross.loc[best, 'P(Survived)']:.4f}")
print(f"存活率最低: Pclass={worst[0]}, Sex={worst[1]} -> {ct_cross.loc[worst, 'P(Survived)']:.4f}")

### 頭等艙女性 vs 三等艙男性：存活率分析

從交叉列聯表可以觀察到：

- **頭等艙 (Pclass=1) 女性**存活率約 **96.8%** — 幾乎全數獲救
- **三等艙 (Pclass=3) 男性**存活率約 **13.5%** — 存活機會極低

這反映了條件機率的**多層條件**效應：

$$P(\text{Survived} | \text{Female}, \text{1st Class}) \gg P(\text{Survived} | \text{Male}, \text{3rd Class})$$

**社會學意涵**：
- 「婦女與兒童優先」原則主要惠及了**上層艙等的女性**
- 三等艙乘客（不分性別）的存活率都明顯偏低
- **社經地位 (Socioeconomic Status)** 與**性別**共同決定了存活機率

> 這正是為什麼在建構預測模型時，我們需要考慮**特徵交互作用 (Feature Interaction)**，
> 而非僅依賴單一特徵的邊際效果。

---
## 重點整理 (Key Takeaways)

| 主題 | 關鍵公式 / 概念 |
|------|------------------|
| 機率三公理 | 非負性、正規性、可加性 |
| 加法法則 | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| 乘法法則 | $P(A \cap B) = P(A) \cdot P(B|A)$ |
| 互斥 vs 獨立 | 互斥 $\neq$ 獨立；互斥指不可同時發生，獨立指互不影響 |
| 條件機率 | $P(A|B) = P(A \cap B) / P(B)$ |
| 貝氏定理 | $P(A|B) = P(B|A) \cdot P(A) / P(B)$ |
| 全機率公式 | $P(A) = \sum P(A|B_i) P(B_i)$ |

---
## 練習題 (Exercises)

### 基礎 (Easy)
1. 計算 $P(\text{Embarked}=S)$，即從南安普敦上船的旅客比例
2. 計算 $P(\text{Survived}=1 | \text{Embarked}=C)$，從瑟堡上船的旅客存活率

### 進階 (Medium)
3. 利用加法法則計算 $P(\text{Pclass}=1 \cup \text{Survived}=1)$
4. 檢驗 `Pclass` 與 `Survived` 是否獨立（比較 $P(A \cap B)$ 與 $P(A) \cdot P(B)$）

### 挑戰 (Hard)
5. 使用貝氏定理計算 $P(\text{Pclass}=1 | \text{Survived}=1)$，並與直接計算結果比對
6. 建立三層交叉表 (`Pclass` x `Sex` x `Embarked`)，找出存活率最高與最低的組合

In [ ]:
# TODO: 練習題 1-2 (Easy)


In [ ]:
# TODO: 練習題 3-4 (Medium)


In [ ]:
# TODO: 練習題 5-6 (Hard)


---

| [< Ch02 常見機率分佈](02_chi-squared-tests.ipynb) | [Ch04 抽樣分佈與中央極限定理 >](04_AB_testing_procedure.ipynb) |
|:---|---:|

---
*Ch03 — 機率基礎與條件機率 | 推論統計學 13 章系列*